# 4a · Does the urban–rural gap explain score instability?

**Research question (RQ 4a):** *By comparing threshold sensitivity at the city level versus national level, this research will test whether urban aggregates remain stable while national scores (which blend urban and rural measurements) show higher volatility.*

## How well can the urban-rural gap be measured with this dataset?

M-Lab city names come from MaxMind's GeoLite2 database, which uses GeoNames names, so most match directly. We match on (country, normalized name) first, then on GeoNames' alternate names for the rest. When several places share a name, we prefer the one in the matching first-level subdivision, then the most populous.

In [ ]:
import zipfile

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon

import iqb_tools as T

T.set_style()
cfg = T.config_table()
tests = T.distinct_tests(
    cfg,
    T.with_thresholds(cfg, T.FRAMEWORK_MIN),
    T.with_thresholds(cfg, T.FRAMEWORK_HIGH),
)
ctx = T.read_table("country_context").set_index("country_code")
MONTHS = pd.date_range("2025-01-01", "2025-12-01", freq="MS")

# National 2025 pass rates per country: the test-weighted average of the 12 months
df25 = T.load_level("country", months=MONTHS)
C = T.pool_pass_rates(df25, T.pass_rates(df25, tests), "country_code", tests)


In [ ]:
GEO = T.EXT / "geonames"
GEO.mkdir(parents=True, exist_ok=True)


def geonames(name):
    """Local copy of a GeoNames dump file (downloaded on first use)."""
    path = GEO / name
    legacy = (
        T.ROOT / "data" / "external" / "geonames" / name
    )  # reuse an earlier raw download if present
    if not path.exists() and legacy.exists():
        return legacy
    if not path.exists():
        url = f"https://download.geonames.org/export/dump/{name.replace('.txt', '.zip') if name == 'cities500.txt' else name}"
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        if name == "cities500.txt":
            (GEO / "cities500.zip").write_bytes(r.content)
            zipfile.ZipFile(GEO / "cities500.zip").extract(name, GEO)
        else:
            path.write_bytes(r.content)
    return path


cols = [
    "geonameid",
    "name",
    "asciiname",
    "alternatenames",
    "lat",
    "lon",
    "fclass",
    "fcode",
    "cc",
    "cc2",
    "admin1",
    "admin2",
    "admin3",
    "admin4",
    "population",
    "elevation",
    "dem",
    "tz",
    "mod",
]
g = pd.read_csv(
    geonames("cities500.txt"),
    sep="\t",
    header=None,
    names=cols,
    quoting=3,
    keep_default_na=False,
    na_values=[""],
    usecols=["name", "asciiname", "alternatenames", "cc", "admin1", "population"],
    dtype={"admin1": str},
)
a1 = pd.read_csv(
    geonames("admin1CodesASCII.txt"),
    sep="\t",
    header=None,
    names=["code", "name", "asciiname", "gid"],
    quoting=3,
    keep_default_na=False,
)
g["admin1_name"] = (g.cc + "." + g.admin1.fillna("")).map(
    dict(zip(a1.code, a1.asciiname.map(T.norm_name)))
)
print(f"{len(g):,} GeoNames places with population ≥ 500")

city = T.load_level("country_city", months=MONTHS)
keys = ["country_code", "subdivision1_name", "city"]
u = city.groupby(keys, dropna=False).n_down.sum().reset_index()
u["nm"] = u.city.map(T.norm_name)
u["sub"] = u.subdivision1_name.fillna("").map(T.norm_name)

primary = pd.concat(
    [g.assign(nm=g.name.map(T.norm_name)), g.assign(nm=g.asciiname.map(T.norm_name))]
)
primary = primary[["cc", "nm", "admin1_name", "population"]].drop_duplicates()
alternate = g[["cc", "alternatenames", "admin1_name", "population"]].dropna(
    subset=["alternatenames"]
)
alternate = alternate.assign(nm=alternate.alternatenames.str.split(",")).explode("nm")
alternate["nm"] = alternate.nm.map(T.norm_name)
alternate = alternate[["cc", "nm", "admin1_name", "population"]].drop_duplicates()


def match(units, ref, how):
    m = units.merge(ref, left_on=["country_code", "nm"], right_on=["cc", "nm"])
    m["same_admin"] = [
        bool(s) and bool(a) and (s == a or s in a or a in s)
        for s, a in zip(m["sub"], m.admin1_name.fillna(""))
    ]
    m = m.sort_values(["same_admin", "population"], ascending=False).drop_duplicates(
        keys
    )
    return m[keys + ["population"]].assign(matched_by=how)


m1 = match(u, primary, "name")
rest = (
    u.merge(m1[keys], how="left", indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)
m2 = match(rest, alternate, "alternate name")
u = u.merge(pd.concat([m1, m2]), on=keys, how="left")
u["matched_by"] = u.matched_by.fillna("unmatched")
print(
    u.groupby("matched_by")
    .agg(cities=("city", "size"), tests=("n_down", "sum"))
    .assign(share_of_tests=lambda x: x.tests / x.tests.sum())
    .round(3)
)
print("\nLargest unmatched (mostly districts of large cities):")
print(
    u[u.matched_by == "unmatched"]
    .nlargest(6, "n_down")[keys + ["n_down"]]
    .to_string(index=False)
)


The first line is the number of GeoNames places used as the reference, every place with at least 500 inhabitants. The table shows how M-Lab's city names were matched to them: *cities* is the number of distinct M-Lab places (country, region and city name), *tests* their download tests in 2025, and *share_of_tests* the share of all tests. Matching by name covers 95.2% of tests, matching by an alternate name another 2.6%, and 2.2% stay unmatched. The list shows the unmatched places with the most tests: mostly districts of large cities (London boroughs, Seoul districts and a district of Cebu City), which have no match among GeoNames' populated places. Unmatched tests are kept apart as *Unmatched* and not assigned a size class.

In [ ]:
CLASSES = ["Large cities", "Other cities", "Towns & rural"]
u["class3"] = pd.cut(u.population, [-1, 50_000, 500_000, np.inf], labels=CLASSES[::-1]).astype(str)
u.loc[u.population.isna(), "class3"] = "Unmatched"
u["class2"] = u.class3.replace({"Large cities": "Cities", "Other cities": "Cities"})
city = city.merge(u[keys + ["population", "class3", "class2"]], on=keys, how="left")

pr = T.pass_rates(city, tests)
# Check: pooling every city row of a country should reproduce the national pass rates
all_cities = T.pool_pass_rates(city, pr, "country_code", tests)
diff = (all_cities[tests.key] - C.loc[all_cities.index, tests.key]).abs().stack()
cover = all_cities.n_down / C.loc[all_cities.index, "n_down"]
print(
    f"pooled cities vs national pass rates: median |diff| {diff.median():.3f}, 90th pct {diff.quantile(0.9):.3f}; "
    f"city table covers a median {cover.median():.0%} of national tests"
)

The check shows that pooling a country's city rows reproduces its national pass rates to within about half a percentage point for the typical test (median difference 0.005, 90th percentile 0.031), and that the city table covers a median 93% of national tests.

In [ ]:
share = city.groupby(["country_code", "class3"]).n_down.sum().unstack(fill_value=0)
share = share.div(share.sum(axis=1), axis=0)
share = share.join(ctx[["income_group", "urban_pct"]]).dropna(subset=["income_group"])
share["rural_pop_pct"] = 100 - share.urban_pct
by_inc = (
    share.groupby("income_group")[CLASSES + ["Unmatched"]]
    .mean()
    .reindex(T.INCOME_ORDER)
)  # mean shares sum to 1
by_inc["rural population share (WDI, national definitions)"] = (
    share.groupby("income_group").rural_pop_pct.mean() / 100
)
by_inc["Towns & rural, median country"] = share.groupby("income_group")[
    "Towns & rural"
].median()
by_inc["countries with < 1% of tests in towns & rural"] = share.groupby(
    "income_group"
)["Towns & rural"].apply(lambda s: (s < 0.01).mean())
by_inc["countries"] = share.groupby("income_group").size()
by_inc.round(3)


Each row is an income group, covering the countries with city data and an income group (*countries*). The first four columns split the average country's 2025 tests by where they are located: *Large cities* (more than 500,000 inhabitants), *Other cities* (50,000 to 500,000), *Towns & rural* (50,000 or fewer) and *Unmatched* (a place name that could not be matched to GeoNames). These four add up to 1. The other columns:

| Column | Meaning |
|---|---|
| *rural population share (WDI, national definitions)* | The average share of the population living in rural areas, from the World Bank, using each country's own definition of rural. |
| *Towns & rural, median country* | The share of tests in towns and rural places for the median country, which a few extreme countries affect less than the average. |
| *countries with < 1% of tests in towns & rural* | The share of countries where towns and rural places have fewer than 1% of the tests. |
| *countries* | The number of countries in the group. |

If tests were located like the population, *Towns & rural* would be close to the rural population share. It is far below it in low-income countries (7% of tests against 62% of the population) and above it in high-income ones (32% against 24%). In more than half of low- and lower-middle-income countries, fewer than 1% of tests are in towns or rural places.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.4))
colors = [T.BLUE_RAMP[k] for k in (700, 450, 250)]  # ordered by settlement size
for j, inc in enumerate(T.INCOME_ORDER):
    left = 0
    for k, c in enumerate(CLASSES):
        v = by_inc.loc[inc, c]
        ax.barh(
            j,
            v - 0.004,
            left=left + 0.002,
            color=colors[k],
            height=0.6,
            label=c if j == 0 else None,
        )
        if v > 0.07:
            ax.text(
                left + v / 2,
                j,
                f"{v:.0%}",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if k < 2 else T.INK,
                zorder=3,
                bbox=dict(facecolor=colors[k], edgecolor="none", pad=1.5),  # keeps the label readable over the tick
            )
        left += v
    rp = by_inc.loc[inc, "rural population share (WDI, national definitions)"]
    ax.plot([1 - rp, 1 - rp], [j - 0.38, j + 0.38], color=T.INK, lw=2, zorder=2)
ax.set_yticks(range(4), [T.INCOME_SHORT[i] for i in T.INCOME_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="y", visible=False)
ax.set_title(
    "Where M-Lab tests are located, 2025 (average country in each group)\n"
    "black tick: if tests followed the population, the rural share would start here"
)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.14))
T.savefig(fig, "4a_test_location_by_income")


The figure shows the first three columns of the table as bars. Each bar is an income group (y-axis), split into the shares of tests located in large cities (darkest), other cities and towns & rural (lightest); the x-axis is the share of the average country's tests, from 0% to 100%. The unmatched tests make up the small gap at the right end of each bar. The black tick marks where the towns-and-rural segment would start if tests were located like the population, at 100% minus the rural population share.

Moving down the bars, the large-city segment shrinks from 74% to 28% and the towns-and-rural segment grows from 7% to 32%. For the three lower income groups the tick sits to the left of the towns-and-rural segment, far to the left for low- and lower-middle-income countries, so rural places have far fewer tests than people. In high-income countries the tick falls inside the towns-and-rural segment.

The table and figure show that rural users are barely visible in M-Lab's city data. In the median low-income country, fewer than 1% of tests are located in towns or rural places, although about 60% of the population is rural. The group average (7%) is pulled up by a few countries such as The Gambia (89%). Uganda (about 790,000 tests in 2025, none located in a town) and Mozambique (about 530,000, none) are typical. The share in towns rises steadily with income (7%, 10%, 22% and 32% by income group). In high-income countries it exceeds the rural population share (32% vs 24%), because many places under 50,000 are suburbs that national definitions count as urban.

This is at least partly a geolocation artifact: it is not plausible that none of Uganda's 790,000 tests came from a town. IP-based location is approximate. Mobile operators often route traffic through a few central points, usually in the capital or largest city, so tests can be placed there even when the user is elsewhere. The data cannot separate mobile from fixed tests, so how much of the gap is location error and how much is testing that really is concentrated in cities cannot be measured in this dataset. 

## Is there an urban–rural gap?

We keep the countries with at least the minimum number of tests (1,153 in 2025, from 1a) in both *Cities* (≥ 50,000 inhabitants) and *Towns & rural* (< 50,000), so that each side's pass rates are known to within ±5 points. Only one low-income country qualifies, so that row describes a single country.

In [ ]:
G = T.pool_pass_rates(city[city.class2 != "Unmatched"], pr[city.class2 != "Unmatched"], ["country_code", "class2"], tests)
SG = T.iqb_scores(G, cfg, T.Q_GRID)
nG = G.n_down.unstack()
nGb = T.binding_tests(G).unstack()  # smaller of download and upload tests
eligible = nGb[(nGb["Cities"] >= T.MIN_TESTS) & (nGb["Towns & rural"] >= T.MIN_TESTS)].index
eligible = eligible.intersection(ctx.index[ctx.income_group.notna()])
d = ctx.loc[eligible].copy()
d["income"] = pd.Categorical(d.income_group, T.INCOME_ORDER, ordered=True)
U = SG.xs("Cities", level="class2").loc[eligible]
R = SG.xs("Towns & rural", level="class2").loc[eligible]
N = T.iqb_scores(C.loc[eligible], cfg, T.Q_GRID)
print(f"{len(d)} countries:", d.income.value_counts(sort=False).to_dict())

gap = pd.DataFrame({f"P{q}": U[q] - R[q] for q in (25, 50, 75, 95)})
summary = gap.groupby(d.income, observed=True).median()
summary["countries"] = d.income.value_counts(sort=False)
summary.loc["All"] = list(gap.median()) + [len(d)]
summary.loc["share with cities ahead"] = list((gap > 0).mean()) + [np.nan]
summary.loc["share with towns ahead"] = list((gap < 0).mean()) + [np.nan]
print("Urban–rural gap = IQB(Cities) − IQB(Towns & rural):")
summary.round(3)

The table gives the urban–rural gap, the IQB score of a country's cities minus that of its towns and rural places, at four percentiles (columns). The first five rows are the median gap for each income group and for all countries, where a positive value means cities score higher. The last two rows are the shares of countries in which cities, or towns, score higher; in the rest, both score the same. For example, at P50 cities score higher in 42.7% of countries and towns in 23.3%, so the two are equal in the remaining third.

The median gap is zero almost everywhere; the exceptions are lower-middle-income countries at P50 (0.047) and the single low-income country. Cities lead most often at P50, towns lead more often than cities at P25, and at P95 both shares are below 8%.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
for inc in T.INCOME_ORDER:
    idx = d.index[d.income == inc]
    if len(idx):
        ax.scatter(R.loc[idx, 50], U.loc[idx, 50], s=24, color=T.INCOME_COLORS[inc],
                   label=f"{T.INCOME_SHORT[inc]} ({len(idx)})", edgecolor=T.SURFACE, linewidth=0.9)
ax.plot([0, 1], [0, 1], color=T.AXIS, lw=1)
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("IQB at P50, towns & rural")
ax.set_ylabel("IQB at P50, cities (≥ 50,000)")
ax.set_title("Cities vs towns & rural, P50, 2025\n(above the line: cities score higher)")
ax.legend(title="income", loc="lower right")
T.savefig(fig, "4a_urban_rural_scatter")

Each dot is one of the 103 countries. The x-axis is its IQB at P50 for towns & rural places, the y-axis its IQB at P50 for cities of 50,000 or more, and the colors are income groups. On the diagonal line the two scores are equal; above it cities score higher, and below it towns do.

Thirty-five dots sit exactly on the line, where cities and towns pass the same tests; 44 are above and 24 below. The largest gaps go both ways: Botswana scores 0.69 in its cities and 0.21 in its towns, while Sri Lanka scores 0.78 in its towns and 0.28 in its cities. High-income countries cluster in the upper right, with cities scoring at least 0.62, and the countries that score low on both sides are lower-middle-, upper-middle- and low-income ones.

The *Cities* class mixes metropolitan cores with mid-sized cities. A sharper contrast is **Large cities (≥ 500,000) vs Towns & rural**, which we compute for the countries meeting the minimum number of tests in each:

In [ ]:
G3 = T.pool_pass_rates(city[city.class3 != "Unmatched"], pr[city.class3 != "Unmatched"], ["country_code", "class3"], tests)
S3 = T.iqb_scores(G3, cfg, [25, 50, 75, 95])
n3b = T.binding_tests(G3).unstack()
el3 = n3b[(n3b["Large cities"] >= T.MIN_TESTS) & (n3b["Towns & rural"] >= T.MIN_TESTS)].index
el3 = el3.intersection(ctx.index[ctx.income_group.notna()])
inc3 = pd.Categorical(ctx.loc[el3, "income_group"], T.INCOME_ORDER, ordered=True)
gap3 = S3.xs("Large cities", level="class3").loc[el3] - S3.xs("Towns & rural", level="class3").loc[el3]
gap3.columns = [f"P{q}" for q in gap3.columns]
t3 = gap3.groupby(inc3, observed=True).median()
t3["countries"] = gap3.groupby(inc3, observed=True).size()
t3.loc["All"] = list(gap3.median()) + [len(el3)]
t3.loc["share with large cities ahead"] = list((gap3 > 0).mean()) + [np.nan]
t3.loc["share with towns ahead"] = list((gap3 < 0).mean()) + [np.nan]
print("Gap = IQB(Large cities) − IQB(Towns & rural):")
t3.round(3)

The table has the same layout as the previous one, with *Large cities* (more than 500,000 inhabitants) in place of all cities, for the 83 countries with at least 1,153 tests in both large cities and towns & rural. The median gap at P50 is larger than with all cities for lower-middle-income (0.088) and upper-middle-income countries (0.129), and large cities lead at P50 in 47% of countries.

Where both sides are observed, the urban–rural gap is modest, concentrated in middle-income countries, and confined to the middle of the distribution.

Comparing all cities with towns & rural, the median country shows **no gap** at any percentile. Lower-middle-income countries show a small one at P50 (0.05). Cities score higher at P50 in about four countries in ten, and towns score higher in about two in ten.

The large-city vs towns contrast is sharper. Metropolitan cores lead more often, mainly in middle-income countries (median gap at P50: 0.09 lower-middle, 0.13 upper-middle), while mid-sized cities often perform like towns.

At **P25**, towns score higher than cities in more countries than the reverse (39 against 22), mostly high-income countries where small-town fixed lines are good. At **P95** the gap is zero for almost nine countries in ten, because the best rural tests meet the same thresholds as the best urban ones. The framework default (P95) therefore cannot see an urban–rural gap.

## Is the national score less stable than the urban one?

We look at two kinds of stability: sensitivity to the percentile (the swing P50 → P95 and the SD across P1–P99), and stability over time (the SD of the monthly score across 2025, with each class re-pooled per month).

In [ ]:
sens = pd.DataFrame({
    ("swing P50→P95", "national"): N[95] - N[50],
    ("swing P50→P95", "cities"): U[95] - U[50],
    ("swing P50→P95", "towns & rural"): R[95] - R[50],
    ("SD P1–P99", "national"): N.std(axis=1, ddof=0),
    ("SD P1–P99", "cities"): U.std(axis=1, ddof=0),
    ("SD P1–P99", "towns & rural"): R.std(axis=1, ddof=0),
})
# Month-to-month: each class re-pooled per month; the national score from each month's own pass rates
Gm = T.pool_pass_rates(city[city.class2 != "Unmatched"], pr[city.class2 != "Unmatched"],
                       ["country_code", "class2", "month"], tests)
vol = T.iqb_scores(Gm, cfg, [50])[50].groupby(level=["country_code", "class2"]).std().unstack()
pr_nat = T.pass_rates(df25, tests).set_index(pd.MultiIndex.from_frame(df25[["country_code", "month"]]))
nat_vol = T.iqb_scores(pr_nat, cfg, [50])[50].groupby(level="country_code").std()
sens[("monthly SD at P50", "national")] = nat_vol.reindex(eligible)
sens[("monthly SD at P50", "cities")] = vol.loc[eligible, "Cities"]
sens[("monthly SD at P50", "towns & rural")] = vol.loc[eligible, "Towns & rural"]

rows = []
for m in ["swing P50→P95", "SD P1–P99", "monthly SD at P50"]:
    x = sens[m].dropna()
    rows.append({"measure": m, **x.median().to_dict(),
                 "national > cities (share)": (x["national"] > x["cities"] + 1e-9).mean(),
                 "national < cities (share)": (x["national"] < x["cities"] - 1e-9).mean(),
                 "Wilcoxon p (national vs cities)": wilcoxon(x["national"], x["cities"]).pvalue})
pd.DataFrame(rows).set_index("measure").round(3)

Each row is one measure of stability, computed for the national score and for the cities and towns-and-rural aggregates of each of the 103 countries; the first three columns are their medians. Lower values mean a more stable score:

| Measure | Meaning |
|---|---|
| *swing P50→P95* | How much the score rises between P50 and P95. |
| *SD P1–P99* | The standard deviation of the score across all percentiles from P1 to P99. |
| *monthly SD at P50* | The standard deviation of the monthly P50 score across the 12 months of 2025. |

The last three columns compare the national score with the cities score, country by country: the share of countries where the national value is higher, the share where it is lower (in the rest they are equal), and the p-value of a Wilcoxon signed-rank test of whether the two differ systematically. A p-value below 0.05 means the difference is unlikely to be chance. Only *SD P1–P99* reaches it (0.001), with medians of 0.339 and 0.336.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
rng = np.random.default_rng(7)  # horizontal jitter only
for ax, m in zip(axes, ["swing P50→P95", "SD P1–P99", "monthly SD at P50"]):
    x = sens[m].dropna()
    for i, (col, c) in enumerate(zip(["cities", "national", "towns & rural"], [T.SERIES[0], T.INK_2, T.SERIES[1]])):
        ax.scatter(i + rng.uniform(-0.15, 0.15, len(x)), x[col], s=14, color=c, alpha=0.7,
                   edgecolor=T.SURFACE, linewidth=0.6)
        ax.hlines(x[col].median(), i - 0.3, i + 0.3, color=T.INK, lw=2)
    ax.set_xticks(range(3), ["cities", "national", "towns &\nrural"])
    ax.set_title(m)
    ax.grid(axis="x", visible=False)
fig.suptitle(f"Stability of cities, national and towns & rural scores ({len(d)} countries; bars: medians)",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
T.savefig(fig, "4a_stability_national_vs_urban")

Each panel shows one measure from the table. Each dot is one country's value for its cities (blue), its national score (gray) or its towns & rural places (orange), spread sideways at random so the dots don't overlap, and the black bars are the medians. The y-axis is the measure's value: higher means less stable.

In every panel the cities and national columns look alike, with medians at nearly the same height. Towns & rural has the highest median swing (0.33) and the highest median monthly SD (0.071). In the swing panel many dots line up at the same few values, because scores move in steps of a test's weight.

National scores are not less stable than urban ones.

For sensitivity to the percentile, the national and cities curves are nearly identical (median swing 0.26 and 0.23). National tests are mostly urban, as the first table showed, so the national score largely *is* the urban score. The only statistically detectable difference, in SD across P1–P99, is negligible (0.339 vs 0.336).

Over time, national and cities scores are equally stable month to month (median SD 0.058 for both). The towns-and-rural aggregate is the least stable (0.071), because it rests on far fewer tests.

So the proposal's hypothesis (urban stable, national volatile because it blends urban and rural) is not supported. Blending does not add instability because there is little rural data to blend in. The instability that does exist (1a, 1b) is in the national *and* the urban scores alike.

### Does a larger urban–rural gap go with a less stable national score?

In [ ]:
rows = []
for m, label in [(("swing P50→P95", "national"), "national swing P50→P95"),
                 (("monthly SD at P50", "national"), "national monthly SD at P50")]:
    y = sens[m]
    rho = spearmanr(gap["P50"], y, nan_policy="omit").statistic
    prho, pp, n = T.partial_spearman(gap["P50"], y, {"lvl": N[50], "n": np.log10(C.loc[eligible, "n_down"])})
    rows.append((label, rho, prho, pp, n))
pd.DataFrame(rows, columns=["outcome", "ρ with gap at P50", "partial ρ (level, log tests)", "partial p", "n"]).round(3)

The table relates the urban–rural gap at P50 to two measures of national instability (rows), across the 103 countries. *ρ with gap at P50* is the plain Spearman rank correlation. *Partial ρ* is the same correlation after removing the part of both that is explained by the national P50 score and the number of tests (on a log scale); *partial p* is its p-value and *n* the number of countries. Values near 0 mean no association.

The gap is not associated with national instability once the national score level and test volume are held fixed (partial ρ 0.03 and 0.14; p = 0.79 and 0.17).

## Does the national score mask a rural score? 


Using the framework's classes from 2a, we ask how often the towns-and-rural aggregate is in a *lower* class than the country as a whole.

In [ ]:
f_min, f_high = (
    T.with_thresholds(cfg, T.FRAMEWORK_MIN),
    T.with_thresholds(cfg, T.FRAMEWORK_HIGH),
)
TIERS = ["Below minimum", "Partial", "Minimum quality", "High quality"]


def classify(pr_, q):
    tmin = pd.DataFrame(
        {uc: T.tipping_percentile(pr_, f_min, uc) for uc in T.USE_CASES}
    )
    thigh = pd.DataFrame(
        {uc: T.tipping_percentile(pr_, f_high, uc) for uc in T.USE_CASES}
    )
    return pd.Series(
        np.select(
            [q > thigh.max(axis=1), q > tmin.max(axis=1), q > tmin.min(axis=1)],
            [3, 2, 1],
            0,
        ),
        index=pr_.index,
    )


Gr = G.xs("Towns & rural", level="class2").loc[eligible]
rows = []
for q in (25, 50, 75, 95):
    nat_c, rur_c = classify(C.loc[eligible], q), classify(Gr, q)
    rows.append(
        {
            "percentile": f"P{q}",
            "towns & rural in a lower class": (rur_c < nat_c).mean(),
            "same class": (rur_c == nat_c).mean(),
            "towns & rural in a higher class": (rur_c > nat_c).mean(),
            "countries": len(eligible),
        }
    )
mask = pd.DataFrame(rows).set_index("percentile")
mask.round(2)


Each row is a percentile, and the values are shares of the 103 countries: the share whose towns-and-rural aggregate falls in a lower framework class than the country as a whole, in the same class, or in a higher class (the classes from 2a: *Below minimum*, *Partial*, *Minimum quality* and *High quality*). Each row adds up to 1. For example, at P50 the towns are in a lower class than their country in 18% of countries and in a higher one in 10%. The classes agree almost always at P25 (99%), least often at P50 (72%), and in 83–84% of countries at P75 and P95.

In [ ]:
w_urban = nG.loc[eligible, "Cities"] / nG.loc[eligible].sum(axis=1)
print(
    f"Cities hold a median {w_urban.median():.0%} of the classified tests in these countries."
)
for q in (50, 95):
    print(
        f"P{q}: national score identical to the cities score for {((N[q] - U[q]).abs() < 1e-9).mean():.0%} of countries, "
        f"to the towns score for {((N[q] - R[q]).abs() < 1e-9).mean():.0%}"
    )


The printout shows why the national score usually matches the urban one. In the median country, cities hold 92% of the tests that could be placed in a city or in a town, so the national pass rates are mostly urban pass rates. At P50 the national score equals the cities score in 78% of countries but the towns score in only 37%. At P95 it equals both in most countries (89% and 85%), because at that percentile cities and towns usually pass the same tests.

The national figure is effectively the urban figure, and it masks a lower rural class in a minority of countries. Cities supply a median 92% of classified tests, so the national score usually coincides with the cities score (78% of countries at P50, 89% at P95). The towns-and-rural aggregate falls in a *lower* class than the country in about one country in six at P50 (18%), and one in seven or eight at P75 and P95 (14% and 12%). The reverse is rarer. At P25 almost every country and its towns are both *below minimum*, so the classes agree for 99% of countries.

## Are individual cities more stable than their country?

We compare individual cities with ≥ 5,000 tests in 2025 with their own country.

In [ ]:
Gc = T.pool_pass_rates(city, pr, keys, tests)
Gc = Gc[Gc.n_down >= 5000]
Sc = T.iqb_scores(Gc, cfg, [50, 95])
cities = Sc.reset_index()
cities["swing"] = cities[95] - cities[50]
cities = cities.merge(u[keys + ["class3", "population"]], on=keys, how="left")
nat = T.iqb_scores(C, cfg, [50, 95])
cities["nat_swing"] = cities.country_code.map(nat[95] - nat[50])
cities["nat_p50"] = cities.country_code.map(nat[50])
cities = cities.merge(ctx[["income_group"]], left_on="country_code", right_index=True, how="left")
tab = cities.groupby("class3").apply(lambda x: pd.Series({
    "cities": len(x),
    "median swing (city)": x.swing.median(),
    "median swing (its country)": x.nat_swing.median(),
    "share less sensitive than country": (x.swing < x.nat_swing - 1e-9).mean(),
    "share more sensitive than country": (x.swing > x.nat_swing + 1e-9).mean(),
    "share scoring below country at P50": (x[50] < x.nat_p50 - 1e-9).mean(),
}), include_groups=False).reindex(CLASSES + ["Unmatched"])
tab.round(2)

Each row is a size class of individual places with at least 5,000 tests in 2025 (*cities* gives how many), and each place is compared with its own country:

| Column | Meaning |
|---|---|
| *median swing (city)* | The median rise in the place's score between P50 and P95. |
| *median swing (its country)* | The same for the countries these places are in. |
| *share less sensitive than country*, *share more sensitive than country* | The share of places whose swing is smaller, or larger, than their country's; for the rest, the two are equal. |
| *share scoring below country at P50* | The share of places whose P50 score is below their country's. |

In every class the shares that are less and more sensitive than their country are similar (37–44%), and 37–46% of places score below their country at P50. Only towns & rural differ from their countries in median swing (0.13 against 0.05), which reflects which countries these towns are in rather than the towns themselves.

Individual cities are about as likely to be more sensitive to the percentile than their country as less (roughly 35–45% each way, in every size class), and roughly four in ten score below their country at P50. No class of city is systematically stable while the national score is volatile. The median swings differ between classes mainly because cities with ≥ 5,000 tests come from different country mixes: 90% of the towns of that size are in high-income countries.

## Save

In [ ]:
out = d[["country", "income_group", "region"]].copy()
out["tests_cities"], out["tests_towns_rural"] = nG.loc[eligible, "Cities"], nG.loc[eligible, "Towns & rural"]
for q in (25, 50, 75, 95):
    out[f"iqb_cities_p{q}"], out[f"iqb_towns_p{q}"], out[f"iqb_national_p{q}"] = U[q], R[q], N[q]
    out[f"gap_p{q}"] = gap[f"P{q}"]
flat = sens.copy()
flat.columns = [f"{a} | {b}" for a, b in flat.columns]
out = out.join(flat.round(4))
out.to_csv(T.TAB / "urban_rural.csv")
loc = share[CLASSES + ["Unmatched", "income_group", "rural_pop_pct"]]
loc.to_csv(T.TAB / "test_location_shares.csv")
cities.to_csv(T.TAB / "city_sensitivity.csv", index=False)
print("saved urban_rural.csv", out.shape, ", test_location_shares.csv", loc.shape, "and city_sensitivity.csv", cities.shape)

## Summary

| Hypothesis / question | Result |
|---|---|
| Can the urban–rural gap be measured with M-Lab data? | **Only partly.** In most low-income countries virtually no tests are geolocated to towns or rural areas. Rural users are under-represented everywhere. |
| Is there an urban–rural gap? | **Modest and selective.** It appears mainly between metropolitan cores and towns, in middle-income countries, at P50. The median country shows none, and there is almost none at P95. |
| Are urban aggregates stable while national scores are volatile? | **No.** National and cities scores are equally sensitive to the percentile and equally stable month to month. |
| Does the gap explain national instability? | **No** association once score level and test volume are held fixed. |
| Does the national score mask rural shortfalls? | **In a minority of countries** (towns & rural in a lower class: 18% at P50, 12% at P95). The national figure is effectively the urban one. |

The national-level instability documented in 1a, 1b and 2a is therefore not an urban–rural blending effect. It comes from where pass rates sit relative to the thresholds and from month-to-month changes in what is measured. For rural policy questions, M-Lab city geolocation is not an adequate basis, especially in low-income countries. That limitation should be stated wherever IQB is used to discuss rural connectivity, and complemented by data with reliable location (e.g. measurements tagged with self-reported or device location).